In [2]:
import os, shutil
from ase.db import connect
from ase.build import add_adsorbate
from ase.io import read, write

db = connect('final_optimized_db.db')
ads_db = connect('Surfaces_Li2S2.db')  # 输出到一个新的中心版数据库

# 你原来给的平面坐标（Å）
local_xy = [(5,5), (6,6), (4,4), (6.5,4.3)]
z_list   = [4.2,   3.24,  3.24,  3.24]
symbols  = ['S',   'Li',  'Li',  'S'   ]

# 分子团几何中心（仅平面 x–y）
mx = sum(x for x, y in local_xy) / len(local_xy)
my = sum(y for x, y in local_xy) / len(local_xy)
offset_xy = [(x - mx, y - my) for (x, y) in local_xy]

def frac_to_xy(atoms, fx, fy):
    # 把分数坐标 (fx, fy) 转为绝对 x–y（考虑非正交晶胞）
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return (float(r[0]), float(r[1]))

for row in db.select():
    atoms = row.toatoms()
    slab = atoms.copy()

    # 目标：表面中心 (0.5, 0.5)
    cx, cy = frac_to_xy(slab, 0.5, 0.5)

    # 整体平移到中心
    for (dx, dy), z, sym in zip(offset_xy, z_list, symbols):
        add_adsorbate(slab, sym, height=z, position=(cx + dx, cy + dy))

    # 防止越界缠绕
    slab.wrap(pbc=[True, True, False])

    ads_db.write(slab, model=row.formula, placement='center')


In [3]:
import os, shutil
from ase.db import connect
from ase.build import add_adsorbate
from ase.io import read, write

db = connect('final_optimized_db.db')
ads_db = connect('Surfaces_Li2S2_bottomright.db')  # 右下角版数据库

local_xy = [(5,5), (6,6), (4,4), (6.5,4.3)]
z_list   = [4.2,   3.24,  3.24,  3.24]
symbols  = ['S',   'Li',  'Li',  'S'   ]

mx = sum(x for x, y in local_xy) / len(local_xy)
my = sum(y for x, y in local_xy) / len(local_xy)
offset_xy = [(x - mx, y - my) for (x, y) in local_xy]

def frac_to_xy(atoms, fx, fy):
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return (float(r[0]), float(r[1]))

margin = 0.30  # 分数坐标边距，可按需调整
for row in db.select():
    atoms = row.toatoms()
    slab = atoms.copy()

    # 目标：右下角（x 靠 1，y 靠 0），各留 margin
    tx, ty = frac_to_xy(slab, 1.0 - margin, margin)

    for (dx, dy), z, sym in zip(offset_xy, z_list, symbols):
        add_adsorbate(slab, sym, height=z, position=(tx + dx, ty + dy))

    slab.wrap(pbc=[True, True, False])

    ads_db.write(slab, model=row.formula, placement='bottom_right', margin=margin)
